In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = pd.read_csv("heart.csv")
data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


In [4]:
data.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,1025.000000,1025.000000,1025.000000,1025.000000,1025.00000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000
mean,54.434146,0.695610,0.942439,131.611707,246.00000,0.149268,0.529756,149.114146,0.336585,1.071512,1.385366,0.754146,2.323902,0.513171
std,9.072290,0.460373,1.029641,17.516718,51.59251,0.356527,0.527878,23.005724,0.472772,1.175053,0.617755,1.030798,0.620660,0.500070
min,29.000000,0.000000,0.000000,94.000000,126.00000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,48.000000,0.000000,0.000000,120.000000,211.00000,0.000000,0.000000,132.000000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,56.000000,1.000000,1.000000,130.000000,240.00000,0.000000,1.000000,152.000000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.000000,1.000000,2.000000,140.000000,275.00000,0.000000,1.000000,166.000000,1.000000,1.800000,2.000000,1.000000,3.000000,1.000000
max,77.000000,1.000000,3.000000,200.000000,564.00000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,4.000000,3.000000,1.000000


In [5]:
data.drop_duplicates(inplace=True)

---

## alpha

In [6]:
alpha = 0.05

## Demographic & Clinical Baselines

Is there a statistically significant difference in the prevalence of heart disease between male and female patients?

In [7]:
data['sex'] = data.sex.map({0: 'female', 1: 'male'})

data['target'] = data.target.map({0: False, 1:True})

In [8]:
sex_crosstab = pd.crosstab(data.target, data.sex)
sex_crosstab

sex,female,male
target,,
False,24,114
True,72,92


In [9]:
from scipy.stats import chi2_contingency

chi2, p_val, dof, expected = chi2_contingency(sex_crosstab)

pd.DataFrame({
    "chi2_stats": [chi2],
    "p-value": [p_val],
    "df": [dof]
})

,chi2_stats,p-value,df
0,23.083879,0.000002,1


---

Does the probability of a heart disease diagnosis increase significantly with age?

$$H_0: \mu_T \leq \mu_F$$
$$H_a: \mu_T > \mu_F$$


In [10]:
disease_ages = data['age'][data['target'] == True]
non_disease_ages = data['age'][data['target'] == False]

In [17]:
from scipy.stats import levene

lev_stats, lev_sign = levene(disease_ages, non_disease_ages)

print(pd.DataFrame({
    "lev_stat": [lev_stats],
    "lev_sign": [lev_sign]
}))

if lev_sign < alpha:
    print("Equal variances not assumed")
else:
    print("Equal variances assumed")

   lev_stat  lev_sign
0  7.634937  0.006079
Equal variances not assumed


In [19]:
from scipy.stats import ttest_ind

tstat1, p_val1 = ttest_ind(disease_ages, non_disease_ages, equal_var=False, alternative='greater')

print(pd.DataFrame({
    "t-statistic": [tstat1],
    "p-value": [p_val1]
}))

if p_val1 < alpha:
    print("Reject the null hypothesis")
else:
    print("Fail to reject the null hypothesis")

   t-statistic   p-value
0    -3.994031  0.999959
Fail to reject the null hypothesis


---

Are higher resting blood pressure (trestbps) or serum cholesterol (chol) levels strong predictors of heart disease in this cohort?

- **Null Hypothesis**: The mean blood pressure in people with heart diseasse is less than or equals those who don't have heart disease in this cohort.

- **Alternative Hypothesis**: The mean of the resting blood pressure in people with heart disease is greater than those who are not diagnosed with a heart disease in this cohort.

$$H_0: \mu_T \leq \mu_F$$
$$H_0: \mu_T > \mu_F$$

In [14]:
hdbp = data["trestbps"][data['target'] == True]
non_hdbp = data["trestbps"][data['target'] == False]

In [15]:
lev_stats1, lev_sign1 = levene(hdbp, non_hdbp)

print(pd.DataFrame({
    "lev_stat": [lev_stats1],
    "lev_sign": [lev_sign1]
}))

if lev_sign1 < alpha:
    print("Equal variances not assumed")
else:
    print("Equal variances assumed")

   lev_stat  lev_sign
0  1.791063   0.18181
Equal variances assumed


In [16]:
tstat2, p_val2 = ttest_ind(a=hdbp,b= non_hdbp, alternative='greater', equal_var=True)

print(pd.DataFrame({
    "t-statistic": [tstat2],
    "p-value": [p_val2]
}))

if p_val2 < alpha:
    print("Reject the null hypothesis")
else:
    print("Fail to reject the null hypothesis")

   t-statistic   p-value
0    -2.560991  0.994537
Fail to reject the null hypothesis


In [20]:
hdchol = data["chol"][data['target'] == True]
non_hdchol = data["chol"][data['target'] == False]

In [29]:
def two_samples_ttest(group1, group2, alternative='two-sided'):
    lev_stats, lev_sign = levene(group1, group2)

    print("Levene's test for variance equality")
    print(20*"-")
    print(pd.DataFrame({
    "lev_stat": [lev_stats],
    "lev_sign": [lev_sign]
    }))

    print('results:')
    if lev_sign < alpha:
        print("Equal variances not assumed")
        var_equal = False
    else:
        print("Equal variances assumed")
        var_equal = True
    
    print(20*'-')
    tstat, p_val = ttest_ind(a=group1,b= group2, alternative=alternative, equal_var=var_equal)

    print("Two-Samples T-test")
    print(20*"-")
    print(pd.DataFrame({
        "t-statistic": [tstat],
        "p-value": [p_val]
    }))

    if p_val < alpha:
        print("Reject the null hypothesis")
    else:
        print("Fail to reject the null hypothesis")

In [30]:
two_samples_ttest(hdchol, non_hdchol, 'greater')

Levene's test for variance equality
--------------------
   lev_stat  lev_sign
0  0.122632  0.726443
results:
Equal variances assumed
--------------------
Two-Samples T-test
--------------------
   t-statistic   p-value
0    -1.415234  0.920982
Fail to reject the null hypothesis


---